# 26 run liana global

**Purpose:** Run global LIANA ligand-receptor inference.

**Original analysis notebook:** `24_run_LIANA_global_main_HVG8000_r0_6.ipynb`

Run this notebook from top to bottom in a fresh kernel. Project paths are
resolved from `GLIOMA_PROJECT_ROOT`; when that variable is not set, the
notebook searches parent directories for the repository root. Generated
AnnData objects are written under `data/processed/` or `data/interim/`,
while tables, figures, and logs retain the workflow's historical output
directories at the repository root.

# 43 — Run LIANA global main CCC analysis (HVG8000 r0.6)

This notebook runs the **main high-confidence global LIANA ligand–receptor analysis** after final annotation and malignant-state consolidation.

Main idea:
- Use final malignant states and high-confidence non-malignant/diploid-leaning lineages.
- Exclude review/ambiguous populations from the main LIANA run.
- Exclude the rare mast-like population from main global analysis because it had low cell support.
- Use log-normalized expression derived from raw `counts`.
- Run LIANA `rank_aggregate` with the consensus ligand–receptor resource.
- Summarize top global interactions, sender–receiver summaries, and malignant/non-malignant communication axes.

Review/exploratory LIANA and condition-specific LIANA should be run in later notebooks.

In [ ]:
# Optional: show environment information
import sys, os, gc, platform, subprocess, textwrap
from pathlib import Path
from datetime import datetime

print('Python:', sys.version)
print('Platform:', platform.platform())
print('Started:', datetime.now().isoformat(timespec='seconds'))


In [ ]:
# Imports
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import scipy.sparse as sp
import scanpy as sc
import anndata as ad
import matplotlib.pyplot as plt

try:
    import liana as li
    LIANA_AVAILABLE = True
    print('[OK] liana imported:', getattr(li, '__version__', 'unknown_version'))
except Exception as e:
    LIANA_AVAILABLE = False
    print('[ERROR] Could not import liana:', repr(e))
    print('Install in your Python environment, for example: pip install liana')

sc.settings.verbosity = 2


In [ ]:
# Paths and configuration
import os

def _find_repository_root(start=None):
    """Locate the repository root or use GLIOMA_PROJECT_ROOT when set."""
    start = Path.cwd() if start is None else Path(start)
    start = start.expanduser().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "README.md").exists() and (candidate / "code").exists():
            return candidate
    return start

PROJECT_DIR = Path(
    os.environ.get("GLIOMA_PROJECT_ROOT", _find_repository_root())
).expanduser().resolve()
METADATA_DIR = PROJECT_DIR / 'metadata'
FIG_DIR = PROJECT_DIR / 'figures' / 'LIANA_HVG8000_r0_6'
OUT_DIR = PROJECT_DIR / "data" / "processed" / "integrated"

METADATA_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

INPUT_H5AD = OUT_DIR / 'GBM_core_4datasets_final_annotation_malignant_states.h5ad'
MAIN_CELLS_TSV = METADATA_DIR / 'LIANA_prepare_HVG8000_r0_6_main_cells.tsv'
INCLUSION_TSV = METADATA_DIR / 'LIANA_prepare_HVG8000_r0_6_celltype_inclusion_decisions.tsv'

OUT_PREFIX = 'LIANA_global_main_HVG8000_r0_6'

# LIANA settings.
GROUPBY = 'liana_celltype_global_main'
COUNTS_LAYER = 'counts'
RESOURCE_NAME = 'consensus'
EXPR_PROP = 0.10
MIN_CELLS = 50
N_PERMS = None  # keeps the run faster; no permutation p-values are computed
N_JOBS = 4
SEED = 1337

# Optional downsampling. Keep None for full main global run.
# If the run is too slow or memory-heavy, set e.g. MAX_CELLS_PER_CELLTYPE = 15000 and rerun.
MAX_CELLS_PER_CELLTYPE = None

# Output filtering settings. Lower LIANA ranks are better.
PRIMARY_RANK_FALLBACKS = ['aggregate_rank', 'magnitude_rank', 'specificity_rank']
RANK_THRESHOLD = 0.05
RANK_THRESHOLD_RELAXED = 0.10
TOP_N_GLOBAL = 1000
TOP_N_PER_SOURCE_TARGET = 30

print('INPUT_H5AD:', INPUT_H5AD)
print('MAIN_CELLS_TSV:', MAIN_CELLS_TSV)
print('FIG_DIR:', FIG_DIR)


In [ ]:
# Helper functions
def write_tsv_replace(df, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists():
        path.unlink()
    df.to_csv(path, sep='\t', index=False)
    print('[WRITE]', path, df.shape)
    return path

def savefig(path, dpi=180):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists():
        path.unlink()
    plt.tight_layout()
    plt.savefig(path, dpi=dpi, bbox_inches='tight')
    plt.close()
    print('[FIG]', path)
    return path

def safe_read_tsv(path, **kwargs):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(str(path))
    return pd.read_csv(path, sep='\t', **kwargs)

def infer_primary_rank(res):
    for c in PRIMARY_RANK_FALLBACKS:
        if c in res.columns:
            return c
    rank_like = [c for c in res.columns if c.endswith('_rank') or 'rank' in c.lower()]
    if rank_like:
        return rank_like[0]
    raise ValueError('No rank-like column found in LIANA results. Columns: ' + ', '.join(map(str, res.columns[:30])))

def clean_str(x):
    return str(x).replace(' ', '_').replace('/', '_')


In [ ]:
# Input readiness checks
readiness_records = []
for name, path in [
    ('input_h5ad', INPUT_H5AD),
    ('main_cells_tsv', MAIN_CELLS_TSV),
    ('inclusion_decisions_tsv', INCLUSION_TSV),
]:
    readiness_records.append({
        'item': name,
        'path': str(path),
        'exists': Path(path).exists(),
        'size_mb': round(Path(path).stat().st_size / 1024 / 1024, 3) if Path(path).exists() else np.nan,
    })

readiness = pd.DataFrame(readiness_records)
write_tsv_replace(readiness, METADATA_DIR / f'{OUT_PREFIX}_readiness.tsv')
display(readiness)

if not LIANA_AVAILABLE:
    raise ImportError('liana is not available in this Python environment. Install liana and rerun this notebook.')
if not INPUT_H5AD.exists() or not MAIN_CELLS_TSV.exists():
    raise FileNotFoundError('Required input file is missing. Check readiness table above.')


In [ ]:
# Read LIANA main cell metadata and inclusion decisions
main_cells = safe_read_tsv(MAIN_CELLS_TSV)
decision = safe_read_tsv(INCLUSION_TSV)

required_main_cols = {'cell_id', 'liana_celltype_main', 'liana_include_main_global'}
missing = required_main_cols - set(main_cells.columns)
if missing:
    raise ValueError(f'Missing columns in main_cells file: {missing}')

main_cells = main_cells[main_cells['liana_include_main_global'].astype(bool)].copy()
# Make sure rare/low-count groups are not included unless inclusion decision says True.
included_types = set(decision.loc[decision['include_main_global'].astype(bool), 'liana_celltype'])
main_cells = main_cells[main_cells['liana_celltype_main'].isin(included_types)].copy()

print('[MAIN CELLS]', main_cells.shape)
print('[MAIN CELL TYPES]', main_cells['liana_celltype_main'].nunique())
display(main_cells['liana_celltype_main'].value_counts().reset_index().rename(columns={'index':'liana_celltype', 'liana_celltype_main':'n_cells'}))

write_tsv_replace(
    main_cells['liana_celltype_main'].value_counts().rename_axis('liana_celltype').reset_index(name='n_cells'),
    METADATA_DIR / f'{OUT_PREFIX}_celltype_counts_used.tsv'
)


In [ ]:
# Load final annotated object and subset to main LIANA cells
adata = sc.read_h5ad(INPUT_H5AD)
adata.var_names_make_unique()
print(adata)

# Map LIANA cell-type labels by cell_id/obs_names.
cell_to_label = main_cells.set_index('cell_id')['liana_celltype_main']
intersect = adata.obs_names.intersection(cell_to_label.index)
print('[INTERSECT cells]', len(intersect), 'of', len(cell_to_label))

adata_liana = adata[intersect].copy()
adata_liana.obs[GROUPBY] = cell_to_label.reindex(adata_liana.obs_names).astype('category')

# Attach key metadata where available.
for col in ['condition', 'sample_id', 'patient_id', 'dataset_id', 'final_annotation_refined', 'final_compartment_major', 'malignant_state', 'copykat_pred_clean']:
    if col in adata.obs.columns:
        adata_liana.obs[col] = adata.obs.loc[adata_liana.obs_names, col].values

# Optional downsampling per LIANA cell type.
if MAX_CELLS_PER_CELLTYPE is not None:
    np.random.seed(SEED)
    keep = []
    for ct, idx in adata_liana.obs.groupby(GROUPBY, observed=True).groups.items():
        idx = np.array(list(idx))
        if len(idx) > MAX_CELLS_PER_CELLTYPE:
            idx = np.random.choice(idx, size=MAX_CELLS_PER_CELLTYPE, replace=False)
        keep.extend(idx.tolist())
    keep = pd.Index(keep)
    adata_liana = adata_liana[keep].copy()
    print('[DOWNSAMPLED]', adata_liana)

print('[LIANA object]', adata_liana)
display(adata_liana.obs[GROUPBY].value_counts().reset_index().rename(columns={'index':'liana_celltype', GROUPBY:'n_cells'}))


In [ ]:
# Prepare log-normalized expression for LIANA from raw counts
if COUNTS_LAYER not in adata_liana.layers:
    raise ValueError(f'Counts layer {COUNTS_LAYER!r} not found. Available layers: {list(adata_liana.layers.keys())}')

# Use raw counts as X, normalize and log1p.
adata_liana.X = adata_liana.layers[COUNTS_LAYER].copy()

# Drop genes with zero expression in this subset to reduce work.
sc.pp.filter_genes(adata_liana, min_cells=1)
print('[After gene filter]', adata_liana)

sc.pp.normalize_total(adata_liana, target_sum=1e4)
sc.pp.log1p(adata_liana)
adata_liana.raw = None

# Basic expression summary.
nnz = adata_liana.X.nnz if sp.issparse(adata_liana.X) else int(np.count_nonzero(adata_liana.X))
print('[X nnz]', nnz)
print('[X shape]', adata_liana.X.shape)


In [ ]:
# Run LIANA rank_aggregate global main analysis
# Notes:
# - Uses consensus LR resource.
# - n_perms=None disables permutation p-values to keep the global run practical.
# - Lower rank values indicate higher priority.

run_config = pd.DataFrame([
    {'parameter': 'groupby', 'value': GROUPBY},
    {'parameter': 'resource_name', 'value': RESOURCE_NAME},
    {'parameter': 'expr_prop', 'value': EXPR_PROP},
    {'parameter': 'min_cells', 'value': MIN_CELLS},
    {'parameter': 'n_perms', 'value': str(N_PERMS)},
    {'parameter': 'n_jobs', 'value': N_JOBS},
    {'parameter': 'max_cells_per_celltype', 'value': str(MAX_CELLS_PER_CELLTYPE)},
    {'parameter': 'n_cells_used', 'value': adata_liana.n_obs},
    {'parameter': 'n_genes_used', 'value': adata_liana.n_vars},
    {'parameter': 'created_at', 'value': datetime.now().isoformat(timespec='seconds')},
])
write_tsv_replace(run_config, METADATA_DIR / f'{OUT_PREFIX}_run_config.tsv')
display(run_config)

li.mt.rank_aggregate(
    adata_liana,
    groupby=GROUPBY,
    resource_name=RESOURCE_NAME,
    expr_prop=EXPR_PROP,
    min_cells=MIN_CELLS,
    use_raw=False,
    layer=None,
    n_perms=N_PERMS,
    n_jobs=N_JOBS,
    seed=SEED,
    verbose=True,
    inplace=True,
    key_added='liana_res',
)

res = adata_liana.uns['liana_res'].copy()
print('[LIANA result]', res.shape)
display(res.head())


In [ ]:
# Standardize and save LIANA results
res = adata_liana.uns['liana_res'].copy()

primary_rank = infer_primary_rank(res)
print('[PRIMARY RANK]', primary_rank)

# Expected columns are source, target, ligand_complex, receptor_complex, plus rank/score columns.
for c in ['source', 'target']:
    if c not in res.columns:
        raise ValueError(f'Missing required LIANA result column: {c}')

if 'ligand_complex' not in res.columns:
    ligand_cols = [c for c in res.columns if 'ligand' in c.lower()]
    res['ligand_complex'] = res[ligand_cols[0]] if ligand_cols else 'unknown_ligand'
if 'receptor_complex' not in res.columns:
    receptor_cols = [c for c in res.columns if 'receptor' in c.lower()]
    res['receptor_complex'] = res[receptor_cols[0]] if receptor_cols else 'unknown_receptor'

res['primary_rank_col'] = primary_rank
res['primary_rank'] = pd.to_numeric(res[primary_rank], errors='coerce')
res['lr_pair'] = res['ligand_complex'].astype(str) + '→' + res['receptor_complex'].astype(str)

def broad_group(x):
    x = str(x)
    if x.startswith('Malignant_'):
        return 'malignant_or_cycling'
    return 'non_malignant_reference_like'

res['source_broad_group'] = res['source'].map(broad_group)
res['target_broad_group'] = res['target'].map(broad_group)
res['communication_axis'] = res['source_broad_group'] + '_to_' + res['target_broad_group']

# Sort by primary rank, lower is stronger.
res = res.sort_values(['primary_rank', 'source', 'target', 'ligand_complex', 'receptor_complex'], ascending=[True, True, True, True, True]).reset_index(drop=True)

# Save full result compressed.
full_path = METADATA_DIR / f'{OUT_PREFIX}_all_results.tsv.gz'
if full_path.exists():
    full_path.unlink()
res.to_csv(full_path, sep='\t', index=False, compression='gzip')
print('[WRITE]', full_path, res.shape)

# Top global.
top_global = res.head(TOP_N_GLOBAL).copy()
write_tsv_replace(top_global, METADATA_DIR / f'{OUT_PREFIX}_top_global_interactions.tsv')

# Rank-threshold interactions, with relaxed fallback.
rank_sig = res[res['primary_rank'] <= RANK_THRESHOLD].copy()
used_threshold = RANK_THRESHOLD
if rank_sig.shape[0] < 100:
    rank_sig = res[res['primary_rank'] <= RANK_THRESHOLD_RELAXED].copy()
    used_threshold = RANK_THRESHOLD_RELAXED
if rank_sig.shape[0] < 100:
    rank_sig = res.head(TOP_N_GLOBAL).copy()
    used_threshold = np.nan
rank_sig['selection_rule'] = f'primary_rank<={used_threshold}' if pd.notna(used_threshold) else f'top_{TOP_N_GLOBAL}_fallback'
write_tsv_replace(rank_sig, METADATA_DIR / f'{OUT_PREFIX}_rank_threshold_or_top_interactions.tsv')

print('[rank selection]', rank_sig.shape, 'threshold:', used_threshold)
display(top_global.head(20))


In [ ]:
# Source-target and axis summaries
# Count and average ranks among prioritized interactions.
rank_for_summary = rank_sig.copy()

source_target_summary = (
    rank_for_summary
    .groupby(['source', 'target', 'source_broad_group', 'target_broad_group', 'communication_axis'], observed=True)
    .agg(
        n_prioritized_interactions=('lr_pair', 'nunique'),
        median_primary_rank=('primary_rank', 'median'),
        min_primary_rank=('primary_rank', 'min'),
    )
    .reset_index()
    .sort_values(['n_prioritized_interactions', 'median_primary_rank'], ascending=[False, True])
)
write_tsv_replace(source_target_summary, METADATA_DIR / f'{OUT_PREFIX}_source_target_summary.tsv')
display(source_target_summary.head(30))

axis_summary = (
    rank_for_summary
    .groupby('communication_axis', observed=True)
    .agg(
        n_prioritized_interactions=('lr_pair', 'nunique'),
        n_source_target_pairs=('source', lambda x: len(set(zip(rank_for_summary.loc[x.index, 'source'], rank_for_summary.loc[x.index, 'target'])))),
        median_primary_rank=('primary_rank', 'median'),
        min_primary_rank=('primary_rank', 'min'),
    )
    .reset_index()
    .sort_values(['n_prioritized_interactions', 'median_primary_rank'], ascending=[False, True])
)
write_tsv_replace(axis_summary, METADATA_DIR / f'{OUT_PREFIX}_axis_summary.tsv')
display(axis_summary)


In [ ]:
# Top interactions per source-target pair
top_per_pair = (
    res
    .groupby(['source', 'target'], observed=True, group_keys=False)
    .head(TOP_N_PER_SOURCE_TARGET)
    .reset_index(drop=True)
)
write_tsv_replace(top_per_pair, METADATA_DIR / f'{OUT_PREFIX}_top_source_target_interactions.tsv')

# Prioritized malignant-related interactions.
malignant_related = rank_sig[
    rank_sig['source_broad_group'].eq('malignant_or_cycling') |
    rank_sig['target_broad_group'].eq('malignant_or_cycling')
].copy()
write_tsv_replace(malignant_related, METADATA_DIR / f'{OUT_PREFIX}_prioritized_malignant_related_interactions.tsv')

# Ligand-receptor pair summary across cell-type pairs.
lr_summary = (
    rank_sig
    .groupby(['ligand_complex', 'receptor_complex', 'lr_pair'], observed=True)
    .agg(
        n_source_target_pairs=('source', lambda x: len(set(zip(rank_sig.loc[x.index, 'source'], rank_sig.loc[x.index, 'target'])))),
        n_sources=('source', 'nunique'),
        n_targets=('target', 'nunique'),
        median_primary_rank=('primary_rank', 'median'),
        min_primary_rank=('primary_rank', 'min'),
    )
    .reset_index()
    .sort_values(['n_source_target_pairs', 'median_primary_rank'], ascending=[False, True])
)
write_tsv_replace(lr_summary, METADATA_DIR / f'{OUT_PREFIX}_ligand_receptor_summary.tsv')
display(lr_summary.head(30))


In [ ]:
# Biological axis labels for easier thesis interpretation
def label_axis(row):
    src = str(row['source'])
    tgt = str(row['target'])
    if src.startswith('Malignant_') and not tgt.startswith('Malignant_'):
        return 'malignant_to_microenvironment'
    if not src.startswith('Malignant_') and tgt.startswith('Malignant_'):
        return 'microenvironment_to_malignant'
    if src.startswith('Malignant_') and tgt.startswith('Malignant_'):
        return 'malignant_state_to_malignant_state'
    return 'microenvironment_to_microenvironment'

rank_sig['biological_axis'] = rank_sig.apply(label_axis, axis=1)

# More focused categories.
def target_lineage(t):
    t = str(t).lower()
    if 'endothelial' in t:
        return 'endothelial'
    if 'mural' in t or 'pericyte' in t or 'fibroblast' in t:
        return 'mural_pericyte_fibroblast'
    if 'oligodendrocyte' in t:
        return 'oligodendrocyte_reference'
    if 'b_plasma' in t or 'plasma' in t:
        return 'B_plasma'
    if 'malignant' in t or 'cycling' in t:
        return 'malignant_state'
    return 'other'

rank_sig['target_lineage_simplified'] = rank_sig['target'].map(target_lineage)
rank_sig['source_lineage_simplified'] = rank_sig['source'].map(target_lineage)

bio_summary = (
    rank_sig
    .groupby(['biological_axis', 'source_lineage_simplified', 'target_lineage_simplified'], observed=True)
    .agg(
        n_interactions=('lr_pair', 'nunique'),
        n_source_target_pairs=('source', lambda x: len(set(zip(rank_sig.loc[x.index, 'source'], rank_sig.loc[x.index, 'target'])))),
        median_primary_rank=('primary_rank', 'median'),
    )
    .reset_index()
    .sort_values(['n_interactions', 'median_primary_rank'], ascending=[False, True])
)
write_tsv_replace(bio_summary, METADATA_DIR / f'{OUT_PREFIX}_biological_axis_summary.tsv')
display(bio_summary.head(50))

# Save annotated rank_sig again with biological columns.
write_tsv_replace(rank_sig, METADATA_DIR / f'{OUT_PREFIX}_prioritized_interactions_annotated.tsv')


In [ ]:
# Figures: source-target heatmaps based on prioritized interactions
figure_records = []

# 1) Interaction count heatmap
pivot_counts = source_target_summary.pivot(index='source', columns='target', values='n_prioritized_interactions').fillna(0)
# Order by total outgoing/incoming for readability
row_order = pivot_counts.sum(axis=1).sort_values(ascending=False).index
col_order = pivot_counts.sum(axis=0).sort_values(ascending=False).index
pivot_counts = pivot_counts.loc[row_order, col_order]

plt.figure(figsize=(max(10, 0.55 * pivot_counts.shape[1]), max(8, 0.45 * pivot_counts.shape[0])))
plt.imshow(pivot_counts.values, aspect='auto')
plt.colorbar(label='Number of prioritized LR interactions')
plt.xticks(range(pivot_counts.shape[1]), pivot_counts.columns, rotation=90, fontsize=8)
plt.yticks(range(pivot_counts.shape[0]), pivot_counts.index, fontsize=8)
plt.title('LIANA global main: prioritized LR interaction counts')
plt.xlabel('Target cell type')
plt.ylabel('Source cell type')
fig_path = savefig(FIG_DIR / f'{OUT_PREFIX}_source_target_prioritized_count_heatmap.png')
figure_records.append({'figure': str(fig_path), 'description': 'Heatmap of number of prioritized LIANA LR interactions by source-target cell type pair.'})

# 2) Median rank heatmap
pivot_rank = source_target_summary.pivot(index='source', columns='target', values='median_primary_rank').reindex(index=row_order, columns=col_order)
plt.figure(figsize=(max(10, 0.55 * pivot_rank.shape[1]), max(8, 0.45 * pivot_rank.shape[0])))
plt.imshow(pivot_rank.values, aspect='auto')
plt.colorbar(label=f'Median {primary_rank} among prioritized interactions')
plt.xticks(range(pivot_rank.shape[1]), pivot_rank.columns, rotation=90, fontsize=8)
plt.yticks(range(pivot_rank.shape[0]), pivot_rank.index, fontsize=8)
plt.title('LIANA global main: median priority rank')
plt.xlabel('Target cell type')
plt.ylabel('Source cell type')
fig_path = savefig(FIG_DIR / f'{OUT_PREFIX}_source_target_median_rank_heatmap.png')
figure_records.append({'figure': str(fig_path), 'description': 'Heatmap of median LIANA primary rank among prioritized interactions by source-target pair. Lower ranks indicate higher priority.'})

# 3) Axis summary barplot
plt.figure(figsize=(10, 5))
axdf = axis_summary.sort_values('n_prioritized_interactions', ascending=True)
plt.barh(axdf['communication_axis'], axdf['n_prioritized_interactions'])
plt.xlabel('Number of prioritized LR interactions')
plt.ylabel('Communication axis')
plt.title('LIANA global main: broad communication axes')
fig_path = savefig(FIG_DIR / f'{OUT_PREFIX}_axis_summary_barplot.png')
figure_records.append({'figure': str(fig_path), 'description': 'Barplot of prioritized LR interaction counts across broad malignant/non-malignant communication axes.'})

figure_manifest = pd.DataFrame(figure_records)
write_tsv_replace(figure_manifest, METADATA_DIR / f'{OUT_PREFIX}_figure_manifest.tsv')
display(figure_manifest)


In [ ]:
# Final report and output manifest
summary_records = [
    {'item': 'n_cells_used', 'value': adata_liana.n_obs},
    {'item': 'n_genes_used', 'value': adata_liana.n_vars},
    {'item': 'n_liana_celltypes', 'value': adata_liana.obs[GROUPBY].nunique()},
    {'item': 'n_liana_results', 'value': res.shape[0]},
    {'item': 'primary_rank_col', 'value': primary_rank},
    {'item': 'n_top_global_saved', 'value': min(TOP_N_GLOBAL, res.shape[0])},
    {'item': 'n_prioritized_or_rank_threshold_interactions', 'value': rank_sig.shape[0]},
    {'item': 'rank_threshold_used', 'value': str(used_threshold)},
    {'item': 'resource_name', 'value': RESOURCE_NAME},
    {'item': 'expr_prop', 'value': EXPR_PROP},
    {'item': 'n_perms', 'value': str(N_PERMS)},
    {'item': 'completed_at', 'value': datetime.now().isoformat(timespec='seconds')},
]
summary = pd.DataFrame(summary_records)
write_tsv_replace(summary, METADATA_DIR / f'{OUT_PREFIX}_run_summary.tsv')
display(summary)

manifest_paths = [
    METADATA_DIR / f'{OUT_PREFIX}_readiness.tsv',
    METADATA_DIR / f'{OUT_PREFIX}_run_config.tsv',
    METADATA_DIR / f'{OUT_PREFIX}_celltype_counts_used.tsv',
    METADATA_DIR / f'{OUT_PREFIX}_all_results.tsv.gz',
    METADATA_DIR / f'{OUT_PREFIX}_top_global_interactions.tsv',
    METADATA_DIR / f'{OUT_PREFIX}_rank_threshold_or_top_interactions.tsv',
    METADATA_DIR / f'{OUT_PREFIX}_top_source_target_interactions.tsv',
    METADATA_DIR / f'{OUT_PREFIX}_source_target_summary.tsv',
    METADATA_DIR / f'{OUT_PREFIX}_axis_summary.tsv',
    METADATA_DIR / f'{OUT_PREFIX}_ligand_receptor_summary.tsv',
    METADATA_DIR / f'{OUT_PREFIX}_biological_axis_summary.tsv',
    METADATA_DIR / f'{OUT_PREFIX}_prioritized_malignant_related_interactions.tsv',
    METADATA_DIR / f'{OUT_PREFIX}_prioritized_interactions_annotated.tsv',
    METADATA_DIR / f'{OUT_PREFIX}_figure_manifest.tsv',
    METADATA_DIR / f'{OUT_PREFIX}_run_summary.tsv',
]
manifest = pd.DataFrame([
    {'path': str(p), 'exists': p.exists(), 'size_mb': round(p.stat().st_size / 1024 / 1024, 3) if p.exists() else np.nan}
    for p in manifest_paths
])
write_tsv_replace(manifest, METADATA_DIR / f'{OUT_PREFIX}_output_manifest.tsv')
display(manifest)

# Clean memory
del adata, adata_liana
gc.collect()


## Notes for interpretation

- LIANA ranks are priority scores where lower rank values generally indicate stronger/more consistent ligand–receptor support.
- This run is a **global main analysis**, not condition-specific. It should be interpreted as the baseline high-confidence CCC landscape.
- Review/ambiguous clusters were excluded from this main run to avoid overinterpreting uncertain populations.
- Condition-specific LIANA should be run after reviewing the global results and applying cell-count filters per condition.
